In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import tensorflow as tf

In [2]:
from keras.layers import Input,Dense,Flatten
from keras.models import Model
from keras.optimizers import Adam
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
from datetime import datetime
from keras.callbacks import ModelCheckpoint
from keras.applications.inception_v3 import InceptionV3,preprocess_input

In [3]:
# Define the image size
IMAGE_SIZE = [299, 299, 3]

# Load the model
inceptionv3 = InceptionV3(include_top=False, input_shape=IMAGE_SIZE, weights='imagenet')

# Visualize the model summary
#inceptionv3.summary()


87910968/87910968 [==============================] - 0s 0us/step


In [4]:
for layer in inceptionv3.layers:
    layer.trainable = False

In [5]:
x = Flatten()(inceptionv3.output)

# Created a new layer as output
prediction = Dense(7, activation='softmax')(x)

# Join it with the model
model = Model(inputs=inceptionv3.input, outputs=prediction)

In [6]:

adam=Adam()

model.compile(loss='categorical_crossentropy',
              optimizer=adam,
              metrics=['accuracy'])

In [7]:
train_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Train'
test_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Test'

In [8]:
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)


# Train data
train_set = train_datagen.flow_from_directory(train_path,
                                              target_size=(299, 299),
                                              batch_size=32,
                                              class_mode='categorical')

# Test data
test_set = test_datagen.flow_from_directory(test_path,
                                            target_size=(299, 299),
                                            batch_size=32,
                                            class_mode='categorical')

Found 6300 images belonging to 7 classes.
Found 1580 images belonging to 7 classes.


In [ ]:
output_layer = model.layers[-1]  # Assuming the output layer is the last layer in the model
num_classes = output_layer.output_shape[-1]  # Number of units in the output layer

print("Number of classes in the output layer:", num_classes)


Number of classes in the output layer: 7


In [9]:
# Define the file name for the model checkpoint
checkpoint_filepath = '/content/drive/MyDrive/Models/inceptionv3.keras'

# Define the ModelCheckpoint callback
checkpoint = ModelCheckpoint(filepath=checkpoint_filepath, verbose=1, save_best_only=True)

# Combine all callbacks
callbacks = [checkpoint]

# Start timing
start = datetime.now()

# Train the model
model_history = model.fit(train_set,
                          validation_data=test_set,
                          epochs=5,
                          steps_per_epoch=197,
                          validation_steps=50,
                          callbacks=callbacks,)

# Calculate duration
duration = datetime.now() - start

print('Total elapsed time:', duration)

Epoch 1/5
197/197 [==============================] - ETA: 0s - loss: 4.5750 - accuracy: 0.7540 
Epoch 1: val_loss improved from inf to 1.49274, saving model to /content/drive/MyDrive/Models/inceptionv3.keras
197/197 [==============================] - 4345s 22s/step - loss: 4.5750 - accuracy: 0.7540 - val_loss: 1.4927 - val_accuracy: 0.8848
Epoch 2/5
197/197 [==============================] - ETA: 0s - loss: 0.7457 - accuracy: 0.9379
Epoch 2: val_loss improved from 1.49274 to 1.35161, saving model to /content/drive/MyDrive/Models/inceptionv3.keras
197/197 [==============================] - 545s 3s/step - loss: 0.7457 - accuracy: 0.9379 - val_loss: 1.3516 - val_accuracy: 0.9127
Epoch 3/5
197/197 [==============================] - ETA: 0s - loss: 0.2803 - accuracy: 0.9717
Epoch 3: val_loss improved from 1.35161 to 1.24437, saving model to /content/drive/MyDrive/Models/inceptionv3.keras
197/197 [==============================] - 504s 3s/step - loss: 0.2803 - accuracy: 0.9717 - val_loss: 1.